
# Kuroshio Daily R1 FIX4
## Speed-Ridge Preserving + Streamline Lookahead Anti-Eddy Qualification

### 为什么不继续让 CNN 直接参与主评分？

R1 FIX2 的结果表明：

- CNN-head 确实降低了 loop day；
- completion 略有提高；
- 但 manual QC 中出现了明显离开最大流速带、进入较弱流区再绕回的路径；
- CNN 分类器的高 AUC 只说明它能区分“baseline 正样本 vs 人工偏移负样本”，并不等价于它能识别真实黑潮主轴。

因此 FIX4 暂停“CNN 直接加权选点”，把**最大流速脊线重新设为硬物理约束**。

### FIX4 核心机制

每一步仍以 characteristic-line 最大流速候选为主：

```text
auxiliary line
    ↓
local speed maxima
    ↓
max-speed candidate = default choice
```

只有当 max-speed candidate 出现明显的“陷涡风险”时，才在**仍属于高流速脊线的 Top-K 候选**之间重排序。

陷涡风险由短程 streamline probe 判断：

```text
candidate
  ↓ follow local u/v
12 km × 5 steps
  ↓
speed persistence
forward progress
directional turning
return-to-history risk
```

因此新算法不是让模型“猜一条路径”，而是：

> **在多个几乎同样强的流速脊候选中，用下游持续性判断哪个更像连续黑潮射流、哪个更像涡旋边缘。**

CNN 在本轮只作为可选消融，不默认参与路径选择。

---

## 本 Notebook 默认只做 Qualification

默认：

```python
RUN_FULL_AFTER_QA = False
```

先跑：

- 原 manual_qc 难例；
- baseline 失败样本；
- baseline 正常样本；

自动比较 FIX2 baseline 和 FIX4。

如果客观 Gate 通过，再人工检查新生成的图。
确认以后把：

```python
RUN_FULL_AFTER_QA = True
```

重新运行 full phase，才生成新的全量 daily dataset 给 R2。


In [ ]:

# ===== 0. 配置 =====
from pathlib import Path
from dataclasses import dataclass, asdict
import os, json, math, random, warnings
import numpy as np
import pandas as pd

@dataclass
class CFG:
    PROJECT_ROOT: str = "/mnt/d/KuroshioDailyR1"

    CACHE50_DIR: str = "cache_50m"
    BASELINE_DIR: str = "tracks_baseline_FIX2_east0"
    OLD_CNN_DIR: str = "tracks_cnn_FIX2_east0"

    FIX4_TRACK_DIR: str = "tracks_physics_FIX4"
    FIX4_REPORT_DIR: str = "R1_FIX4/reports"
    FIX4_FIGURE_DIR: str = "R1_FIX4/figures/manual_qc"
    FIX4_DATASET_DIR: str = "R1_FIX4/dataset"

    MANIFEST_FILE: str = "reports/daily_manifest.csv"
    BASELINE_REPORT: str = "reports/baseline_tracker_report_FIX2.csv"
    OLD_CNN_REPORT: str = "reports/cnn_tracker_report_FIX2.csv"
    MANUAL_QC_TEMPLATE: str = "reports/manual_qc_template.csv"

    # 原tracker
    SEED_LON_MIN: float = 121.0
    SEED_LON_MAX: float = 122.0
    SEED_LAT_MIN: float = 22.0
    SEED_LAT_MAX: float = 23.0
    TERMINAL_LON: float = 146.0
    EXTENSION_LON: float = 141.0

    AUX_HALF_KM: float = 70.0
    AUX_POINTS: int = 21

    STEP_MAIN_KM: float = 10.0
    STEP_RESCUE_KM: float = 20.0
    STEP_EXTENSION_KM: float = 50.0
    MAX_TRACK_POINTS: int = 1000

    REPEAT_THRESHOLD_KM: float = 1.5
    RESCUE_ANGLE_DEG: float = 30.0
    RESCUE_STEPS: int = 20

    # FIX4：高流速脊硬约束
    TOP_K: int = 5
    SPEED_FLOOR_NORMAL: float = 0.85
    SPEED_FLOOR_RESCUE: float = 0.72

    # streamline probe
    PROBE_STEP_KM: float = 12.0
    PROBE_STEPS: int = 5
    PROBE_HISTORY_GUARD_KM: float = 25.0
    PROBE_HISTORY_SKIP_RECENT: int = 10

    # 只有max-speed候选存在风险时才rerank
    RISK_PROGRESS_MIN: float = 0.45
    RISK_PERSISTENCE_MIN: float = 0.55
    RISK_ALIGN_MIN: float = 0.55
    RISK_TURN_MAX_DEG: float = 80.0

    # rerank分数。speed始终占绝对主导。
    W_SPEED: float = 0.58
    W_PERSIST: float = 0.18
    W_PROGRESS: float = 0.14
    W_ALIGN: float = 0.10
    LOOP_PENALTY: float = 0.30
    TURN_PENALTY: float = 0.10

    # QA
    QA_RANDOM_GOOD: int = 24
    QA_RANDOM_FAIL: int = 24
    RANDOM_SEED: int = 20260925

    # 客观资格门槛
    MAX_REGRESS_COMPLETE_TO_FAIL: int = 1
    MIN_MEDIAN_RIDGE_RATIO: float = 0.90
    MIN_P10_RIDGE_RATIO: float = 0.75
    MAX_LOW_RIDGE_FRAC: float = 0.05
    MAX_P95_TURN_DEGRADATION: float = 5.0

    # 默认只跑小样本资格验证
    RUN_FULL_AFTER_QA: bool = False

cfg = CFG()

ROOT = Path(cfg.PROJECT_ROOT)
CACHE50 = ROOT / cfg.CACHE50_DIR
BASE_DIR = ROOT / cfg.BASELINE_DIR
OLD_CNN_DIR = ROOT / cfg.OLD_CNN_DIR

FIX4_TRACK = ROOT / cfg.FIX4_TRACK_DIR
REPORT = ROOT / cfg.FIX4_REPORT_DIR
FIG = ROOT / cfg.FIX4_FIGURE_DIR
DATASET = ROOT / cfg.FIX4_DATASET_DIR

for p in [FIX4_TRACK, REPORT, FIG, DATASET]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(cfg.RANDOM_SEED)
np.random.seed(cfg.RANDOM_SEED)

print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))


In [ ]:

# ===== 1. 输入文件检查 + 读取FIX2结果 =====
MANIFEST_PATH = ROOT / cfg.MANIFEST_FILE
BASE_REPORT_PATH = ROOT / cfg.BASELINE_REPORT
CNN_REPORT_PATH = ROOT / cfg.OLD_CNN_REPORT
MANUAL_QC_PATH = ROOT / cfg.MANUAL_QC_TEMPLATE

for p in [CACHE50, BASE_DIR, MANIFEST_PATH, BASE_REPORT_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"缺少必要输入: {p}")

manifest = pd.read_csv(MANIFEST_PATH, parse_dates=["date"])
base_report = pd.read_csv(BASE_REPORT_PATH, parse_dates=["date"])

cnn_report = None
if CNN_REPORT_PATH.exists():
    cnn_report = pd.read_csv(CNN_REPORT_PATH, parse_dates=["date"])

print("manifest rows:", len(manifest))
print("baseline days:", len(base_report))
print("baseline completion:", base_report["completed"].mean())
print("baseline loop-day rate:", (base_report["loop_events"].fillna(0)>0).mean())

if cnn_report is not None:
    print("old CNN completion:", cnn_report["completed"].mean())
    print("old CNN loop-day rate:", (cnn_report["loop_events"].fillna(0)>0).mean())


In [ ]:

# ===== 2. 几何 / 插值 / 物理工具 =====
from scipy.interpolate import RegularGridInterpolator
from scipy.signal import find_peaks

EARTH_R_KM = 6371.0088

def wrap_pi(a):
    return (a + np.pi) % (2*np.pi) - np.pi

def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.asarray, (lon1,lat1,lon2,lat2))
    p1, p2 = np.deg2rad(lat1), np.deg2rad(lat2)
    dl = np.deg2rad(lon2-lon1)
    dp = p2-p1
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*EARTH_R_KM*np.arcsin(np.minimum(1.0,np.sqrt(a)))

def angle_east_rad(lon1,lat1,lon2,lat2):
    p1,p2 = np.deg2rad(lat1),np.deg2rad(lat2)
    dl = np.deg2rad(lon2-lon1)
    y = np.sin(dl)*np.cos(p2)
    x = np.cos(p1)*np.sin(p2)-np.sin(p1)*np.cos(p2)*np.cos(dl)
    nav = np.arctan2(y,x)
    return wrap_pi(np.pi/2-nav)

def forward_point(lon,lat,east_angle,distance_km):
    nav = np.pi/2-east_angle
    phi1,lam1 = np.deg2rad(lat),np.deg2rad(lon)
    d = distance_km/EARTH_R_KM
    phi2 = np.arcsin(
        np.sin(phi1)*np.cos(d) +
        np.cos(phi1)*np.sin(d)*np.cos(nav)
    )
    lam2 = lam1 + np.arctan2(
        np.sin(nav)*np.sin(d)*np.cos(phi1),
        np.cos(d)-np.sin(phi1)*np.sin(phi2)
    )
    return float(np.rad2deg(lam2)),float(np.rad2deg(phi2))

def aux_line(lon,lat,flow_dir,half_km=None,n=None):
    half_km = cfg.AUX_HALF_KM if half_km is None else half_km
    n = cfg.AUX_POINTS if n is None else n
    ds = np.linspace(-half_km,half_km,n)
    xs,ys = [],[]
    for s in ds:
        b = flow_dir + (np.pi/2 if s>=0 else -np.pi/2)
        x,y = forward_point(lon,lat,b,abs(float(s)))
        xs.append(x); ys.append(y)
    return np.asarray(xs),np.asarray(ys),ds

def matlab_aux_weights(n):
    k=np.arange(1,n+1,dtype=float)
    return 1.0-np.abs((n+1)/2.0-k)/(n-1)

def make_interpolators(lon,lat,u,v):
    return (
        RegularGridInterpolator((lat,lon),u,bounds_error=False,fill_value=np.nan),
        RegularGridInterpolator((lat,lon),v,bounds_error=False,fill_value=np.nan),
    )

def interp_uv(ui,vi,lons,lats):
    pts=np.column_stack([lats,lons])
    return ui(pts),vi(pts)

def interp_one(ui,vi,lon,lat):
    uu = float(ui([[lat,lon]])[0])
    vv = float(vi([[lat,lon]])[0])
    return uu,vv

# smoke
e = forward_point(130,30,0,10)
n = forward_point(130,30,np.pi/2,10)
assert e[0]>130 and abs(e[1]-30)<0.05
assert n[1]>30 and abs(n[0]-130)<0.05
print("[geometry] PASS")


In [ ]:

# ===== 3. Streamline probe：判断“连续射流”还是“闭合涡边缘” =====
def streamline_probe(
    ui,vi,
    lon0,lat0,
    reference_dir,
    candidate_speed,
    history_lon,
    history_lat
):
    x=float(lon0)
    y=float(lat0)

    speeds=[]
    dirs=[]
    points=[(x,y)]
    loop_risk=False

    for _ in range(cfg.PROBE_STEPS):
        uu,vv=interp_one(ui,vi,x,y)

        if not np.isfinite(uu) or not np.isfinite(vv):
            break

        sp=float(np.hypot(uu,vv))
        if sp < 1e-6:
            break

        d=float(np.arctan2(vv,uu))
        speeds.append(sp)
        dirs.append(d)

        x,y=forward_point(
            x,y,d,cfg.PROBE_STEP_KM
        )
        points.append((x,y))

        # 检查短程streamline是否返回历史轨迹
        if (
            len(history_lon) >
            cfg.PROBE_HISTORY_SKIP_RECENT
        ):
            hx=np.asarray(
                history_lon[:-cfg.PROBE_HISTORY_SKIP_RECENT]
            )
            hy=np.asarray(
                history_lat[:-cfg.PROBE_HISTORY_SKIP_RECENT]
            )
            dd=haversine_km(hx,hy,x,y)
            if (
                len(dd) and
                np.isfinite(dd).any() and
                np.nanmin(dd)<cfg.PROBE_HISTORY_GUARD_KM
            ):
                loop_risk=True

    if not speeds:
        return {
            "persistence":0.0,
            "progress":0.0,
            "alignment":0.0,
            "turn_penalty":1.0,
            "loop_risk":True,
            "probe_n":0,
        }

    speeds=np.asarray(speeds,float)
    dirs=np.asarray(dirs,float)

    # 相对候选点流速的持续性
    persistence=float(np.clip(
        np.mean(speeds)/(candidate_speed+1e-6),
        0.0,1.0
    ))

    # 净位移投影到当前下游方向 / 理论probe总长度
    x1,y1=points[-1]
    net_km=float(
        haversine_km(lon0,lat0,x1,y1)
    )
    net_dir=float(
        angle_east_rad(lon0,lat0,x1,y1)
    )
    projected=max(
        0.0,
        net_km*np.cos(
            wrap_pi(net_dir-reference_dir)
        )
    )
    total_possible=max(
        1e-6,
        cfg.PROBE_STEP_KM*len(speeds)
    )
    progress=float(np.clip(
        projected/total_possible,
        0.0,1.0
    ))

    # 流向与参考下游方向的一致性
    alignment=float(np.clip(
        np.mean(
            (np.cos(wrap_pi(dirs-reference_dir))+1.0)/2.0
        ),
        0.0,1.0
    ))

    if len(dirs)>=2:
        turns=np.abs(
            np.rad2deg(
                wrap_pi(np.diff(dirs))
            )
        )
        turn_penalty=float(np.clip(
            np.mean(turns)/90.0,
            0.0,1.0
        ))
    else:
        turn_penalty=0.5

    return {
        "persistence":persistence,
        "progress":progress,
        "alignment":alignment,
        "turn_penalty":turn_penalty,
        "loop_risk":bool(loop_risk),
        "probe_n":int(len(speeds)),
    }


In [ ]:

# ===== 4. FIX4 candidate selector：最大流速默认，只有风险时才rerank =====
def candidate_peak_indices(speed,valid):
    s=np.asarray(speed,float).copy()
    s[~valid]=-np.inf

    good=np.where(valid)[0]
    if len(good)==0:
        return []

    global_max=int(good[np.argmax(s[good])])

    s_for_peaks=np.where(valid,s,-1e9)
    peaks,_=find_peaks(s_for_peaks)

    ids=set([global_max])
    for p in peaks:
        if valid[p]:
            ids.add(int(p))

    # 再补充top speed点，避免采样过稀时find_peaks漏掉峰
    top = good[np.argsort(s[good])[::-1][:cfg.TOP_K]]
    ids.update(map(int,top))

    return sorted(
        ids,
        key=lambda k:s[k],
        reverse=True
    )[:max(cfg.TOP_K,1)]

def select_candidate_fix4(
    ui,vi,
    center_lon,center_lat,
    mean_dir,
    history_lon,history_lat,
    rescue=False
):
    xl,yl,_=aux_line(
        center_lon,center_lat,mean_dir
    )
    uc,vc=interp_uv(ui,vi,xl,yl)
    speed=np.hypot(uc,vc)

    valid=(
        np.isfinite(speed) &
        np.isfinite(uc) &
        np.isfinite(vc)
    )
    if not valid.any():
        return None

    max_speed=float(np.nanmax(speed[valid]))
    if max_speed<=0:
        return None

    # rescue阶段继续保留原方向约束
    if rescue and len(history_lon):
        prev_lon=float(history_lon[-1])
        prev_lat=float(history_lat[-1])

        geom_dirs=np.array([
            angle_east_rad(
                prev_lon,prev_lat,x,y
            )
            for x,y in zip(xl,yl)
        ])

        geom_diff=np.abs(
            wrap_pi(geom_dirs-mean_dir)
        )

        constrained=(
            valid &
            (geom_diff<np.deg2rad(cfg.RESCUE_ANGLE_DEG))
        )

        if constrained.any():
            valid=constrained

    peaks=candidate_peak_indices(
        speed,valid
    )
    if not peaks:
        return None

    # global max永远是默认候选
    k0=peaks[0]

    def eval_k(k):
        sp=float(speed[k])
        ratio=float(sp/(max_speed+1e-8))

        flow_dir=float(
            np.arctan2(vc[k],uc[k])
        )

        flow_align=float(
            (np.cos(
                wrap_pi(flow_dir-mean_dir)
            )+1.0)/2.0
        )

        probe=streamline_probe(
            ui,vi,
            float(xl[k]),float(yl[k]),
            mean_dir,
            sp,
            history_lon,history_lat
        )

        score=(
            cfg.W_SPEED*ratio +
            cfg.W_PERSIST*probe["persistence"] +
            cfg.W_PROGRESS*probe["progress"] +
            cfg.W_ALIGN*flow_align -
            cfg.LOOP_PENALTY*float(probe["loop_risk"]) -
            cfg.TURN_PENALTY*probe["turn_penalty"]
        )

        return {
            "k":int(k),
            "lon":float(xl[k]),
            "lat":float(yl[k]),
            "u":float(uc[k]),
            "v":float(vc[k]),
            "speed":sp,
            "speed_ratio":ratio,
            "flow_align":flow_align,
            "score":float(score),
            **probe
        }

    base=eval_k(k0)

    # max-speed流脊没有明显风险时，绝不让其他模型/规则把它换掉
    base_flow_dir=float(
        np.arctan2(vc[k0],uc[k0])
    )
    immediate_turn=float(
        abs(np.rad2deg(
            wrap_pi(base_flow_dir-mean_dir)
        ))
    )

    risky=(
        base["loop_risk"] or
        base["progress"]<cfg.RISK_PROGRESS_MIN or
        base["persistence"]<cfg.RISK_PERSISTENCE_MIN or
        base["flow_align"]<cfg.RISK_ALIGN_MIN or
        immediate_turn>cfg.RISK_TURN_MAX_DEG
    )

    if not risky:
        base["reranked"]=False
        base["baseline_risky"]=False
        return base

    # 只有此时才从“仍然属于高流速脊”的候选里选择
    floor=(
        cfg.SPEED_FLOOR_RESCUE
        if rescue else
        cfg.SPEED_FLOOR_NORMAL
    )

    candidates=[]
    for k in peaks:
        e=eval_k(k)
        if e["speed_ratio"]>=floor:
            candidates.append(e)

    if not candidates:
        base["reranked"]=False
        base["baseline_risky"]=True
        return base

    best=max(
        candidates,
        key=lambda d:d["score"]
    )
    best["reranked"]=(
        best["k"]!=base["k"]
    )
    best["baseline_risky"]=True

    return best


In [ ]:

# ===== 5. FIX4 tracker =====
def _seed_point(lon,lat,u,v):
    ix=np.where(
        (lon>=cfg.SEED_LON_MIN)&
        (lon<=cfg.SEED_LON_MAX)
    )[0]
    iy=np.where(
        (lat>=cfg.SEED_LAT_MIN)&
        (lat<=cfg.SEED_LAT_MAX)
    )[0]

    if len(ix)==0 or len(iy)==0:
        raise RuntimeError("seed region outside grid")

    su=u[np.ix_(iy,ix)]
    sv=v[np.ix_(iy,ix)]
    sp2=su**2+sv**2

    if not np.isfinite(sp2).any():
        raise RuntimeError("seed region has no valid velocity")

    j,i=np.unravel_index(
        np.nanargmax(sp2),
        sp2.shape
    )

    lon0=float(lon[ix[i]])
    lat0=float(lat[iy[j]])
    theta=float(
        np.arctan2(sv[j,i],su[j,i])
    )
    return lon0,lat0,theta

def track_fix4(lon,lat,u,v):
    ui,vi=make_interpolators(
        lon,lat,u,v
    )

    lon0,lat0,theta0=_seed_point(
        lon,lat,u,v
    )

    path_lon=[]
    path_lat=[]
    dirs=[]

    ridge_ratios=[]
    probe_progress=[]
    probe_persistence=[]
    reranked_flags=[]
    risky_flags=[]

    loop_events=0
    rescue_remaining=0
    stop_reason="max_points"

    center_lon,center_lat=lon0,lat0
    current_dir=theta0

    for _ in range(cfg.MAX_TRACK_POINTS):
        xl,yl,_=aux_line(
            center_lon,
            center_lat,
            current_dir
        )
        uu,vv=interp_uv(
            ui,vi,xl,yl
        )

        valid=(
            np.isfinite(uu)&
            np.isfinite(vv)
        )
        if not valid.any():
            stop_reason="no_valid_aux_flow"
            break

        w=matlab_aux_weights(len(xl))
        den=np.sum(w[valid])
        if den<=0:
            stop_reason="bad_aux_weight"
            break

        mean_u=np.sum(
            uu[valid]*w[valid]
        )/den
        mean_v=np.sum(
            vv[valid]*w[valid]
        )/den
        mean_dir=float(
            np.arctan2(mean_v,mean_u)
        )

        cand=select_candidate_fix4(
            ui,vi,
            center_lon,center_lat,
            mean_dir,
            path_lon,path_lat,
            rescue=(rescue_remaining>0)
        )

        if cand is None:
            stop_reason="no_candidate"
            break

        cl,ca=cand["lon"],cand["lat"]

        repeat_idx=None
        if len(path_lon)>8:
            hx=np.asarray(path_lon[:-5])
            hy=np.asarray(path_lat[:-5])
            dd=haversine_km(
                hx,hy,cl,ca
            )
            if (
                len(dd) and
                np.isfinite(dd).any() and
                np.nanmin(dd)<cfg.REPEAT_THRESHOLD_KM
            ):
                repeat_idx=int(
                    np.nanargmin(dd)
                )

        if repeat_idx is not None:
            loop_events+=1

            path_lon=path_lon[:repeat_idx+1]
            path_lat=path_lat[:repeat_idx+1]
            dirs=dirs[:repeat_idx+1]

            ridge_ratios=ridge_ratios[:repeat_idx+1]
            probe_progress=probe_progress[:repeat_idx+1]
            probe_persistence=probe_persistence[:repeat_idx+1]
            reranked_flags=reranked_flags[:repeat_idx+1]
            risky_flags=risky_flags[:repeat_idx+1]

            if loop_events>=3 or len(path_lon)<2:
                stop_reason="too_many_loops"
                break

            current_dir=dirs[-1]

            step_km=(
                cfg.STEP_EXTENSION_KM
                if path_lon[-1]>cfg.EXTENSION_LON
                else cfg.STEP_RESCUE_KM
            )

            center_lon,center_lat=forward_point(
                path_lon[-1],
                path_lat[-1],
                current_dir,
                step_km
            )

            rescue_remaining=cfg.RESCUE_STEPS
            continue

        path_lon.append(cl)
        path_lat.append(ca)
        dirs.append(mean_dir)

        ridge_ratios.append(
            cand["speed_ratio"]
        )
        probe_progress.append(
            cand["progress"]
        )
        probe_persistence.append(
            cand["persistence"]
        )
        reranked_flags.append(
            float(cand["reranked"])
        )
        risky_flags.append(
            float(cand["baseline_risky"])
        )

        if cl>=cfg.TERMINAL_LON:
            stop_reason="terminal_lon"
            break

        step_km=(
            cfg.STEP_EXTENSION_KM
            if cl>cfg.EXTENSION_LON
            else cfg.STEP_MAIN_KM
        )

        center_lon,center_lat=forward_point(
            cl,ca,mean_dir,step_km
        )
        current_dir=mean_dir
        rescue_remaining=max(
            0,rescue_remaining-1
        )

    x=np.asarray(path_lon,dtype=np.float32)
    y=np.asarray(path_lat,dtype=np.float32)

    completed=bool(
        len(x)>2 and
        np.isfinite(x).any() and
        np.nanmax(x)>=cfg.TERMINAL_LON
    )

    return {
        "lon":x,
        "lat":y,
        "direction":np.asarray(dirs,dtype=np.float32),
        "ridge_ratio":np.asarray(ridge_ratios,dtype=np.float32),
        "probe_progress":np.asarray(probe_progress,dtype=np.float32),
        "probe_persistence":np.asarray(probe_persistence,dtype=np.float32),
        "reranked":np.asarray(reranked_flags,dtype=np.float32),
        "risky":np.asarray(risky_flags,dtype=np.float32),
        "completed":completed,
        "loop_events":int(loop_events),
        "n_points":int(len(x)),
        "stop_reason":stop_reason,
    }

def track_metrics(tr):
    x=np.asarray(tr["lon"])
    y=np.asarray(tr["lat"])

    out={
        "completed":bool(tr["completed"]),
        "loop_events":int(tr["loop_events"]),
        "n_points":int(tr["n_points"]),
        "stop_reason":tr["stop_reason"],
        "max_lon":float(np.nanmax(x)) if len(x) else np.nan,
        "final_lon":float(x[-1]) if len(x) else np.nan,
        "final_lat":float(y[-1]) if len(y) else np.nan,
    }

    if len(x)>1:
        out["path_km"]=float(
            np.nansum(
                haversine_km(
                    x[:-1],y[:-1],
                    x[1:],y[1:]
                )
            )
        )
    else:
        out["path_km"]=0.0

    if len(x)>3:
        dirs=np.array([
            angle_east_rad(
                x[i],y[i],
                x[i+1],y[i+1]
            )
            for i in range(len(x)-1)
        ])
        turns=np.abs(
            np.rad2deg(
                wrap_pi(np.diff(dirs))
            )
        )
        out["p95_turn_deg"]=float(
            np.nanpercentile(turns,95)
        )
    else:
        out["p95_turn_deg"]=np.nan

    rr=np.asarray(tr["ridge_ratio"])
    out["ridge_median"]=float(np.nanmedian(rr)) if len(rr) else np.nan
    out["ridge_p10"]=float(np.nanpercentile(rr,10)) if len(rr) else np.nan
    out["ridge_low_frac"]=float(np.mean(rr<0.75)) if len(rr) else np.nan
    out["rerank_frac"]=float(np.mean(np.asarray(tr["reranked"])>0)) if len(rr) else np.nan
    out["risky_frac"]=float(np.mean(np.asarray(tr["risky"])>0)) if len(rr) else np.nan

    return out

def save_fix4_track(tr,path):
    np.savez_compressed(
        path,
        **{
            k:v for k,v in tr.items()
            if k!="stop_reason"
        },
        stop_reason=np.array(
            tr["stop_reason"]
        ),
        tracker_version=np.array(
            "R1_FIX4_PHYSICS_RIDGE_LOOKAHEAD"
        )
    )


In [ ]:

# ===== 6. 构建QA日期集 =====
manual_dates=[]

if MANUAL_QC_PATH.exists():
    q=pd.read_csv(MANUAL_QC_PATH)
    if "date" in q.columns:
        manual_dates=list(
            pd.to_datetime(
                q["date"]
            ).dropna().dt.normalize().unique()
        )

good=base_report[
    base_report["completed"] &
    (base_report["loop_events"]==0)
]["date"].dropna().tolist()

fail=base_report[
    ~base_report["completed"]
]["date"].dropna().tolist()

rng=np.random.default_rng(
    cfg.RANDOM_SEED
)

random_good=list(
    rng.choice(
        good,
        size=min(
            cfg.QA_RANDOM_GOOD,
            len(good)
        ),
        replace=False
    )
) if good else []

random_fail=list(
    rng.choice(
        fail,
        size=min(
            cfg.QA_RANDOM_FAIL,
            len(fail)
        ),
        replace=False
    )
) if fail else []

qa_dates=sorted(set(
    list(pd.to_datetime(manual_dates)) +
    list(pd.to_datetime(random_good)) +
    list(pd.to_datetime(random_fail))
))

# 只保留有50m cache的日期
qa_dates=[
    d for d in qa_dates
    if (
        CACHE50 /
        f"{pd.Timestamp(d):%Y%m%d}.npz"
    ).exists()
]

print("manual difficult dates:",len(manual_dates))
print("random baseline-good:",len(random_good))
print("random baseline-fail:",len(random_fail))
print("QA total unique dates:",len(qa_dates))

pd.DataFrame({
    "date":qa_dates
}).to_csv(
    REPORT/"qa_dates.csv",
    index=False
)


In [ ]:

# ===== 7. 运行FIX4 QA =====
rows=[]

for dt in qa_dates:
    dt=pd.Timestamp(dt)
    z=np.load(
        CACHE50/f"{dt:%Y%m%d}.npz"
    )

    tr=track_fix4(
        z["lon"],
        z["lat"],
        z["u"],
        z["v"]
    )

    met=track_metrics(tr)
    met["date"]=dt
    rows.append(met)

    save_fix4_track(
        tr,
        FIX4_TRACK/f"{dt:%Y%m%d}.npz"
    )

qa_new=pd.DataFrame(rows).sort_values("date")

cmp=(
    base_report.merge(
        qa_new,
        on="date",
        how="inner",
        suffixes=("_base","_fix4")
    )
)

print("QA compared days:",len(cmp))

qa_new.to_csv(
    REPORT/"qa_fix4_report.csv",
    index=False,
    encoding="utf-8-sig"
)

cmp.to_csv(
    REPORT/"qa_baseline_vs_fix4.csv",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:

# ===== 8. 客观Gate =====
regress = int(
    (
        cmp["completed_base"] &
        (~cmp["completed_fix4"])
    ).sum()
)

improve = int(
    (
        (~cmp["completed_base"]) &
        cmp["completed_fix4"]
    ).sum()
)

completion_base=float(
    cmp["completed_base"].mean()
)
completion_fix4=float(
    cmp["completed_fix4"].mean()
)

loop_base=float(
    (
        cmp["loop_events_base"]>0
    ).mean()
)
loop_fix4=float(
    (
        cmp["loop_events_fix4"]>0
    ).mean()
)

turn_base=float(
    cmp["p95_turn_deg_base"].median()
)
turn_fix4=float(
    cmp["p95_turn_deg_fix4"].median()
)

ridge_median=float(
    cmp["ridge_median"].median()
)
ridge_p10=float(
    cmp["ridge_p10"].median()
)
low_frac=float(
    cmp["ridge_low_frac"].mean()
)

gate_rows=[
    {
        "metric":"completion_rate",
        "baseline":completion_base,
        "fix4":completion_fix4,
    },
    {
        "metric":"loop_day_rate",
        "baseline":loop_base,
        "fix4":loop_fix4,
    },
    {
        "metric":"median_p95_turn_deg",
        "baseline":turn_base,
        "fix4":turn_fix4,
    },
    {
        "metric":"median_ridge_ratio",
        "baseline":np.nan,
        "fix4":ridge_median,
    },
    {
        "metric":"median_p10_ridge_ratio",
        "baseline":np.nan,
        "fix4":ridge_p10,
    },
    {
        "metric":"mean_low_ridge_fraction",
        "baseline":np.nan,
        "fix4":low_frac,
    },
]

gate_table=pd.DataFrame(gate_rows)
display(gate_table)

checks={
    "regress_complete_to_fail":
        regress<=cfg.MAX_REGRESS_COMPLETE_TO_FAIL,

    "completion_not_worse":
        completion_fix4>=completion_base,

    "loop_not_worse":
        loop_fix4<=loop_base,

    "ridge_median":
        ridge_median>=cfg.MIN_MEDIAN_RIDGE_RATIO,

    "ridge_p10":
        ridge_p10>=cfg.MIN_P10_RIDGE_RATIO,

    "low_ridge_fraction":
        low_frac<=cfg.MAX_LOW_RIDGE_FRAC,

    "turn_not_materially_worse":
        turn_fix4<=(
            turn_base+
            cfg.MAX_P95_TURN_DEGRADATION
        ),
}

print("\ncompletion fail→complete:",improve)
print("completion complete→fail:",regress)
print("\nGate checks:")
for k,v in checks.items():
    print(k,":",v)

AUTO_GATE_PASS=all(checks.values())
print("\nAUTO_GATE_PASS =",AUTO_GATE_PASS)

gate_table.to_csv(
    REPORT/"qa_gate_summary.csv",
    index=False
)

with open(
    REPORT/"qa_gate.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "qa_days":int(len(cmp)),
            "improved_fail_to_complete":improve,
            "regressed_complete_to_fail":regress,
            "checks":checks,
            "auto_gate_pass":AUTO_GATE_PASS,
        },
        f,
        indent=2,
        ensure_ascii=False
    )


In [ ]:

# ===== 9. 新manual QC图：baseline / old-CNN / FIX4 =====
import matplotlib.pyplot as plt

def load_track_any(path):
    z=np.load(path)
    return {
        "lon":z["lon"],
        "lat":z["lat"],
    }

# 优先：原24个manual难例 + FIX4最可疑日期
cmp["fix4_suspicion"]=(
    (~cmp["completed_fix4"]).astype(float)*5 +
    cmp["loop_events_fix4"].fillna(0)*3 +
    (1-cmp["ridge_p10"].fillna(0))*3 +
    cmp["ridge_low_frac"].fillna(1)*3 +
    cmp["p95_turn_deg_fix4"].fillna(180)/90
)

worst_fix4=cmp.sort_values(
    "fix4_suspicion",
    ascending=False
).head(24)["date"].tolist()

plot_dates=sorted(set(
    list(pd.to_datetime(manual_dates)) +
    list(pd.to_datetime(worst_fix4))
))

for dt in plot_dates:
    dt=pd.Timestamp(dt)
    cache=CACHE50/f"{dt:%Y%m%d}.npz"
    bp=BASE_DIR/f"{dt:%Y%m%d}.npz"
    fp=FIX4_TRACK/f"{dt:%Y%m%d}.npz"
    cp=OLD_CNN_DIR/f"{dt:%Y%m%d}.npz"

    if not (
        cache.exists() and
        bp.exists() and
        fp.exists()
    ):
        continue

    z=np.load(cache)
    sp=np.hypot(
        z["u"],
        z["v"]
    )

    base=load_track_any(bp)
    fix4=load_track_any(fp)
    oldcnn=load_track_any(cp) if cp.exists() else None

    rr=qa_new.loc[
        qa_new["date"]==dt
    ].iloc[0] if (qa_new["date"]==dt).any() else None

    fig,ax=plt.subplots(
        figsize=(11,7.5)
    )

    im=ax.pcolormesh(
        z["lon"],
        z["lat"],
        sp,
        shading="auto"
    )

    ax.plot(
        base["lon"],
        base["lat"],
        lw=1.2,
        label="FIX2 baseline"
    )

    if oldcnn is not None:
        ax.plot(
            oldcnn["lon"],
            oldcnn["lat"],
            lw=1.0,
            alpha=0.75,
            label="old CNN-head"
        )

    ax.plot(
        fix4["lon"],
        fix4["lat"],
        lw=1.8,
        label="FIX4 physics-ridge"
    )

    if len(fix4["lon"]):
        ax.scatter(
            fix4["lon"][-1],
            fix4["lat"][-1],
            s=32,
            marker="x",
            label="FIX4 end"
        )

    ax.set_xlim(120,148)
    ax.set_ylim(20,42)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    if rr is not None:
        ax.set_title(
            f"{dt.date()} | "
            f"complete={bool(rr.completed)} "
            f"loops={int(rr.loop_events)} "
            f"ridge_med={rr.ridge_median:.3f} "
            f"ridge_p10={rr.ridge_p10:.3f}"
        )
    else:
        ax.set_title(str(dt.date()))

    ax.legend(
        loc="best",
        fontsize=8
    )
    fig.colorbar(
        im,
        ax=ax,
        label="speed (m/s)"
    )
    fig.tight_layout()

    fig.savefig(
        FIG/f"{dt:%Y%m%d}_FIX4_compare.png",
        dpi=180
    )
    plt.close(fig)

print("manual QC figures ->",FIG)



## 10. 人工 Gate

即使 `AUTO_GATE_PASS=True`，也不要仅凭 completion / loop rate 判定流轴“正确”。

请重点看新生成的：

```text
D:\KuroshioDailyR1\R1_FIX4\figures\manual_qc\
```

尤其检查：

1. 台湾东侧是否紧贴强流核；
2. 东海陆坡段是否持续位于强流带；
3. 吐噶喇海峡后是否进入正确主流；
4. 日本南侧大蛇行是否沿强流脊，而不是切过低速区；
5. KE 是否继续沿主流强流带向东；
6. FIX4 是否修掉原 CNN 在约 130–140°E 的低速区绕行。

如果这些图仍明显错误，不要进入 full phase。


In [ ]:

# ===== 11. Full phase：默认关闭 =====
if not cfg.RUN_FULL_AFTER_QA:
    print(
        "RUN_FULL_AFTER_QA=False\n"
        "本轮到此停止。先检查QA报告和manual_qc图。\n"
        "确认物理路径正确后，把配置改成 True 再运行本单元。"
    )
elif not AUTO_GATE_PASS:
    raise RuntimeError(
        "AUTO_GATE_PASS=False，不允许进入全量路径构建。"
    )
else:
    full_rows=[]

    available_dates=(
        manifest.loc[
            manifest["cache_status"].isin(
                ["cached","written"]
            ),
            "date"
        ]
        .dropna()
        .sort_values()
        .tolist()
    )

    from tqdm.auto import tqdm

    for dt in tqdm(
        available_dates,
        desc="FIX4 full tracker"
    ):
        dt=pd.Timestamp(dt)
        cache=CACHE50/f"{dt:%Y%m%d}.npz"
        out=FIX4_TRACK/f"{dt:%Y%m%d}.npz"

        try:
            z=np.load(cache)
            tr=track_fix4(
                z["lon"],
                z["lat"],
                z["u"],
                z["v"]
            )
            save_fix4_track(
                tr,out
            )
            met=track_metrics(tr)
            met["date"]=dt
            full_rows.append(met)

        except Exception as e:
            full_rows.append({
                "date":dt,
                "completed":False,
                "loop_events":np.nan,
                "n_points":0,
                "stop_reason":"exception",
                "error":repr(e),
            })

    full_report=pd.DataFrame(
        full_rows
    ).sort_values("date")

    full_report.to_csv(
        REPORT/"full_tracker_report_FIX4.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "full completion:",
        full_report["completed"].mean()
    )
    print(
        "full loop-day:",
        (
            full_report["loop_events"].fillna(0)>0
        ).mean()
    )
    print(
        "full ridge median:",
        full_report["ridge_median"].median()
    )


In [ ]:

# ===== 12. Full通过后重新生成双分支逐日数据 =====
if cfg.RUN_FULL_AFTER_QA and AUTO_GATE_PASS:
    from scipy.interpolate import interp1d

    TARGET_LATS_1=np.arange(
        22.0,
        25.5+1e-9,
        1/12
    )
    TARGET_LONS_2=np.arange(
        122.0,
        145.0+1e-9,
        1/12
    )

    def branch1_from_track(lons,lats):
        valid=(
            np.isfinite(lons)&
            np.isfinite(lats)&
            (lats<=25.7)
        )
        lons=np.asarray(lons)[valid]
        lats=np.asarray(lats)[valid]

        if len(lats)<2:
            return np.full(
                len(TARGET_LATS_1),
                np.nan,
                dtype=np.float32
            )

        df=(
            pd.DataFrame({
                "lat":lats,
                "lon":lons
            })
            .groupby("lat",as_index=False)
            .lon.mean()
            .sort_values("lat")
        )

        if len(df)<2:
            return np.full(
                len(TARGET_LATS_1),
                np.nan,
                dtype=np.float32
            )

        f=interp1d(
            df.lat.values,
            df.lon.values,
            bounds_error=False,
            fill_value="extrapolate"
        )

        return f(
            TARGET_LATS_1
        ).astype(np.float32)

    def branch2_from_track(lons,lats):
        valid=(
            np.isfinite(lons)&
            np.isfinite(lats)
        )
        lons=np.asarray(lons)[valid]
        lats=np.asarray(lats)[valid]

        if len(lons)<2:
            return np.full(
                len(TARGET_LONS_2),
                np.nan,
                dtype=np.float32
            )

        # 对相同经度聚合；
        # <=125E保留较北分支，之后使用均值，与旧流程兼容
        vals=[]
        for lo in np.sort(
            np.unique(lons)
        ):
            yy=lats[lons==lo]
            vals.append(
                (
                    lo,
                    np.max(yy)
                    if lo<=125
                    else np.mean(yy)
                )
            )

        arr=np.asarray(vals,float)
        if len(arr)<2:
            return np.full(
                len(TARGET_LONS_2),
                np.nan,
                dtype=np.float32
            )

        # FIX4输出用于qualification；
        # 使用按经度线性插值，避免路径拼接过程再次引入truth信息。
        f=interp1d(
            arr[:,0],
            arr[:,1],
            bounds_error=False,
            fill_value="extrapolate"
        )

        return f(
            TARGET_LONS_2
        ).astype(np.float32)

    date_list=[]
    b1_list=[]
    b2_list=[]

    for r in full_report.itertuples(
        index=False
    ):
        if not bool(r.completed):
            continue

        p=FIX4_TRACK/f"{pd.Timestamp(r.date):%Y%m%d}.npz"
        if not p.exists():
            continue

        z=np.load(p)
        b1=branch1_from_track(
            z["lon"],z["lat"]
        )
        b2=branch2_from_track(
            z["lon"],z["lat"]
        )

        if (
            np.isfinite(b1).all() and
            np.isfinite(b2).all()
        ):
            date_list.append(
                pd.Timestamp(r.date)
            )
            b1_list.append(b1)
            b2_list.append(b2)

    B1=np.stack(b1_list)
    B2=np.stack(b2_list)

    out_path=(
        DATASET/
        "daily_dual_branch_paths_FIX4.npz"
    )

    np.savez_compressed(
        out_path,
        dates=np.array([
            d.strftime("%Y-%m-%d")
            for d in date_list
        ]),
        branch1_lon=B1,
        branch1_target_lats=TARGET_LATS_1,
        branch2_lat=B2,
        branch2_target_lons=TARGET_LONS_2,
    )

    print(
        "FIX4 daily dataset:",
        out_path,
        "N=",
        len(date_list)
    )


In [ ]:

# ===== 13. Handoff =====
handoff={
    "tracker":"R1_FIX4_PHYSICS_RIDGE_LOOKAHEAD",
    "run_full":bool(cfg.RUN_FULL_AFTER_QA),
    "qa_days":int(len(cmp)),
    "qa_auto_gate_pass":bool(AUTO_GATE_PASS),
    "qa_baseline_completion":completion_base,
    "qa_fix4_completion":completion_fix4,
    "qa_baseline_loop_rate":loop_base,
    "qa_fix4_loop_rate":loop_fix4,
    "qa_ridge_median":ridge_median,
    "qa_ridge_p10":ridge_p10,
    "qa_low_ridge_fraction":low_frac,
    "qa_improved_fail_to_complete":improve,
    "qa_regressed_complete_to_fail":regress,
}

if (
    cfg.RUN_FULL_AFTER_QA and
    "full_report" in globals()
):
    handoff.update({
        "full_days":int(len(full_report)),
        "full_completion":float(
            full_report["completed"].mean()
        ),
        "full_loop_rate":float(
            (
                full_report["loop_events"].fillna(0)>0
            ).mean()
        ),
        "full_ridge_median":float(
            full_report["ridge_median"].median()
        ),
    })

with open(
    REPORT/"R1_FIX4_HANDOFF.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        handoff,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    json.dumps(
        handoff,
        indent=2,
        ensure_ascii=False
    )
)
